# EDA Experiments
Use this notebook to experiment with the profiler, issue detector, and cleaning pipeline.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import plotly.io as pio
pio.renderers.default = 'notebook'

from core.pipeline import CleaningPipeline
from utils.visualization import *
from utils.metrics import compute_quality_metrics, quality_score

In [ ]:
# ── Generate a synthetic dirty dataset ───────────────────────────────────────
rng = np.random.default_rng(42)
n = 500

df_dirty = pd.DataFrame({
    'id':       range(n),
    'age':      rng.integers(18, 80, n).astype(float),
    'income':   rng.normal(50_000, 15_000, n),
    'score':    rng.uniform(0, 100, n),
    'category': rng.choice(['A', 'B', 'C', None], n),
    'note':     ['  hello  ' if i % 10 == 0 else 'world' for i in range(n)],
    'constant': [1] * n,
})

# Inject missing values
df_dirty.loc[rng.choice(n, 80, replace=False), 'age']    = np.nan
df_dirty.loc[rng.choice(n, 60, replace=False), 'income'] = np.nan
df_dirty.loc[rng.choice(n, 40, replace=False), 'score']  = np.nan

# Inject duplicates
df_dirty = pd.concat([df_dirty, df_dirty.iloc[:25]], ignore_index=True)

# Inject outliers
df_dirty.loc[:5, 'income'] = 2_000_000
df_dirty.loc[:3, 'age']    = 200

print(f'Shape: {df_dirty.shape}')
df_dirty.head()

In [ ]:
# ── Run the full pipeline ─────────────────────────────────────────────────────
import io

buf = io.StringIO()
df_dirty.to_csv(buf, index=False)
buf.seek(0)

pipeline = CleaningPipeline(config_path='../config/config.yaml')
result   = pipeline.run(buf, file_name='experiment.csv', save_output=False)

print('Issues:', len(result['issues']))
print('Actions:', len(result['actions']))
print('Changes applied:', len(result['change_log']))

In [ ]:
# ── Visualise quality scores ──────────────────────────────────────────────────
qs = result['report']['quality_score']
print(f"Quality: {qs['before']} → {qs['after']} (+{qs['improvement']} pts)")

plot_quality_gauge(qs['before'], 'Before').show()
plot_quality_gauge(qs['after'],  'After').show()

In [ ]:
# ── EDA plots ─────────────────────────────────────────────────────────────────
plot_missing_bar(result['df_raw']).show()
plot_numeric_distributions(result['df_raw']).show()
plot_outlier_boxplots(result['df_raw']).show()
plot_before_after_comparison(result['report']['delta']).show()

In [ ]:
# ── Inspect change log ────────────────────────────────────────────────────────
import pandas as pd
pd.DataFrame(result['change_log'])